## SparkSession:

SparkSession is the `entry point` to Apache Spark. It used to `create DataFrames, read files, run SQL queries, and perform data processing `using Spark.

In [0]:
from pyspark.sql import SparkSession

In [0]:
spark=SparkSession.builder.appName("Spark").getOrCreate()

In [0]:
print(spark)

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Spark") \
    .getOrCreate()

print("Spark Session Created Successfully!")
print(spark.version)

## A. Create a DataFrame:

In [0]:
data=[
    (1,"abc",10000),
    (2,"def",20000),
    (3,"ghi",30000)
]

columns=["id","name","salary"]

df=spark.createDataFrame(data,columns)
df.show()

## B. Read a CSV file:

`header=True and inferSchema=True` are used when reading a CSV file. They tell Spark how to interpret the first row and the column data types.

In [0]:
df=spark.read.csv(
    "/Workspace/Data-Enginnering/Data-Engineering-Python/PySpark-DataBricks/Source/employee.csv",
    header=True,
    inferSchema=True
)

df.show()

## C. Execute SQL queries:

`createOrReplaceTempView()` is used to give a `temporary name for table DataFrame` then only we can query it using SQL.

In [0]:
df.createOrReplaceTempView("employee")

spark.sql("""
    SELECT FirstName, BranchID
    FROM employee
""").show()

In [0]:
df.printSchema()

print(df.columns)

## D.Data Processing:

Data Processing means `cleaning, transforming, filtering, and preparing raw data for analysis`. This is one of the main tasks of a Data Engineer.

## Step 1: Read the source data:


| Code | Meaning |
|---|---|
| `header=True` | The first row contains column names. |
| `inferSchema=True` | Spark attempts to automatically detect data types. |
| `df.show()` | Displays the first 20 rows by default. |


In [0]:
df = spark.read.csv(
    "/Workspace/Data-Enginnering/Data-Engineering-Python/PySpark-DataBricks/Source/employee.csv",
    header=True,
    inferSchema=True
)

df.show()

header=True: first row contains column names.
inferSchema=True: Spark attempts to detect data types.
df.show(): displays the first 20 rows by default.

## Step 2: Inspect the data:

Check the `column names, data types, row count, and sample records` before changing anything.

In [0]:
# Display first 20 rows
df.show()

# Display column names
print(df.columns)

# Display data types
df.printSchema()

# Count rows
print("Total rows:", df.count())

# Display 5 rows without truncating long values
df.show(5, truncate=False)

## Step 3: Check missing values:

Find `how many null values exist` in each column.

In [0]:
from pyspark.sql.functions import col, sum, when

df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).show()

## Step 4: Remove duplicate records:

`dropDuplicates()` removes rows that are identical across all columns.

In [0]:
df = df.dropDuplicates()

print("Rows after removing duplicates:", df.count())

## Step 5: Handle missing values:

A. To `remove rows` where `all columns are null`:

In [0]:
from pyspark.sql.functions import col

df = df.dropna(how="all")

B. To `remove rows` where a `particular important column is null`, for example `EmployeeID`:

In [0]:
df = df.filter(col("EmployeeID").isNotNull())

## Step 6: Transform the Data

For the employee dataset, the columns include `FirstName`, `LastName`, `Position`, and `Hiredate`.

### Difference between `col()` and `withColumn()`

| Feature | `col()` | `withColumn()` |
|---|---|---|
| Purpose | References an existing column | Adds a new column or replaces an existing column |
| Used for | Selecting, filtering, and transforming column values | Modifying the DataFrame structure |
| Import | `from pyspark.sql.functions import col` | No separate import needed |
| Returns | A Column expression | A new DataFrame |
| Example | `col("FirstName")` | `df.withColumn("FullName", col("FirstName"))` |


In [0]:
from pyspark.sql.functions import trim, concat_ws, to_date

# Remove extra spaces from employee names
df = df.withColumn("FirstName", trim(col("FirstName")))
df = df.withColumn("LastName", trim(col("LastName")))

# Create a full name column
df = df.withColumn(
    "FullName",
    concat_ws(" ", col("FirstName"), col("LastName"))
)

# Inspect the transformed data
df.show(truncate=False)

If Hiredate is stored as a string, convert it to a date using the correct format

## Step 7: Filter records:

For example, find `employees whose position is Manager`.

In [0]:
df.filter(
    col("Position") == "Manager"
).show()

## Step 8: Perform aggregation:

`Count employees by position`:

In [0]:
df.groupBy("Position") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

## Step 9: Register a temporary view and run SQL:

In [0]:
df.createOrReplaceTempView("employees")

spark.sql("""
    SELECT Position, COUNT(*) AS total_employees
    FROM employees
    GROUP BY Position
    ORDER BY total_employees DESC
""").show()

## Step 10: Save the processed data:

In [0]:
output_path = "/Volumes/workspace/default/data_volume/employees"

df.write \
    .mode("overwrite") \
    .parquet(output_path)

print("Data saved successfully:", output_path)

%md
## Step 11: Read the processed data:

In [0]:
processed_df = spark.read.parquet(
    "/Volumes/workspace/default/data_volume/employees"
)

processed_df.show(truncate=False)

In [0]:
processed_df.printSchema()

print("Total processed rows:", processed_df.count())

## Understand your pipeline
- Source: `/Workspace/ApacheSpark/Source` — your input CSV data.
- Processing: `PySpark cleans, transforms, and analyzes the data`.
- Output: `/Volumes/workspace/default/data_volume/employees` — your processed Parquet data.